# Results summary — every table from committed JSON

Regenerates the paper's tables (§5 Robustness battery and headline) purely from the
small committed `eval/results/**.json` files — **no multi-GB activation tensors required**.
This decouples "paper numbers" from "raw artifacts": a fresh single-digit-MB clone can
reproduce every table. Run top-to-bottom from the repo root.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

# Locate repo root (the dir containing eval/results), from cwd or a parent.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'eval'/'results').exists()), Path.cwd())
def load(rel):
    p = ROOT/rel
    return json.load(open(p)) if p.exists() else None
print('repo root:', ROOT)

## §5.1 Training-induced variance (multi-seed)

In [ ]:
s = load('eval/results/multiseed/multiseed_summary.json')
rows = []
for cfg, agg in (s['configs'].items() if s else []):
    d = agg['delta_sae_over_raw']
    rows.append({'config': cfg, 'Δ(SAE−Raw) mean': round(d['mean'],4), 'std': round(d['std'],4),
                 'min': round(d['min'],4), 'max': round(d['max'],4)})
pd.DataFrame(rows)

## §5.2 SAE design space (sweep + transcoder)

In [ ]:
sw, tc = load('eval/results/sweep/sweep_summary.json'), load('eval/results/sweep/transcoder_summary.json')
print('Autoencoder sweep:', sw['n_cells'], 'cells,', sw['n_positive_delta_sig'],
      'significantly positive; max Δ(SAE−Raw)=', round(sw['max_delta_sae_over_raw'],4))
if tc:
    print('Transcoder:', tc['n_cells'], 'cells,', tc['n_positive_delta_sig'],
          'significantly positive; max Δ=', round(tc['max_delta_sae_over_raw'],4))

## §5.3 Δ(SAE−Raw) vs depth (all 24 layers)

In [ ]:
s = load('eval/results/layersweep/layersweep_summary.json')
df = pd.DataFrame(sorted(s['cells'], key=lambda c: c['layer']))[['layer','P4_RawOnly_AUROC','P5_SAEOnly_AUROC','delta_sae_over_raw']] if s else pd.DataFrame()
print(s['n_layers_sae_beats_raw_sig'], 'of', len(s['cells']), 'layers significantly positive') if s else None
from IPython.display import Image, display
fig = ROOT/'eval/results/layersweep/layersweep_summary.png'
display(Image(str(fig))) if fig.exists() else None
df

## §5.4 Frontier SAE (Gemma Scope on Gemma-2-2B)

In [ ]:
r = load('eval/results/gemma/gemma_probe_results.json')
pd.DataFrame([{ 'P4 RawOnly': round(r['P4_RawOnly_AUROC'],4), 'P5 SAEOnly': round(r['P5_SAEOnly_AUROC'],4),
  'Δ(SAE−Raw)': round(r['delta_sae_over_raw'],4),
  'CI': [round(r['delta_sae_over_raw_CI_lower'],4), round(r['delta_sae_over_raw_CI_upper'],4)] }]) if r else 'missing'

## §5.5–5.7 Second backbone (Qwen), ARC-Easy, TriviaQA

In [ ]:
s = load('eval/results/extension/extension_summary.json')
df = pd.DataFrame([{ 'config': c['label'], 'test_hard_rate': round(c['test_hard_rate'],3),
  'best_AUROC': round(c['best_AUROC'],3), 'Δ(SAE−Raw)': round(c['delta_sae_over_raw'],4),
  'CI': [round(x,3) for x in c['delta_sae_over_raw_CI']] } for c in s['cells']]) if s else pd.DataFrame()
print('configs where SAE significantly beats raw:', s.get('n_sae_beats_raw_sig')) if s else None
df

## §5.8 Stronger baseline (weak vs strong P1)

In [ ]:
rows = []
for lyr in ['squad_l12','squad_l18']:
    d = load(f'eval/results/strong_baseline/{lyr}.json')
    if d: rows.append({'config': lyr, 'P1 weak': round(d['weak']['P1_AUROC'],3),
        'P1 strong': round(d['strong']['P1_AUROC'],3), 'P1 gain': round(d['P1_gain_from_strong'],4),
        'strong Δ(SAE−Raw)': round(d['strong']['delta_sae_over_raw'],4), 'null holds': d['null_holds_strong']})
pd.DataFrame(rows)

## §5.9 Coverage power analysis + position-matched disentangle

In [ ]:
s = load('eval/results/coverage/coverage_power.json')
df = pd.DataFrame([{ 'K': int(k), 'recon_penalty': round(v['delta_recon_natural'],4),
  'detected/5': v['n_features_detected'], 'mean_power': round(v['mean_power'],3) }
  for k, v in sorted(s['by_k'].items(), key=lambda kv: int(kv[0]))]) if s else pd.DataFrame()
print('coverage for 80% power ~', round(s['coverage_for_80pct_power'],2), 'positions') if s else None
from IPython.display import Image, display
fig = ROOT/'eval/results/coverage/coverage_power.png'
display(Image(str(fig))) if fig.exists() else None
pm = load('eval/results/position_matched/summary.json')
if pm: print('position-matched: fidelity(B−A)=', round(pm['fidelity_effect_B_minus_A'],4),
             ' coverage(C−B)=', round(pm['coverage_effect_C_minus_B'],4))
df

## §5.10 Causal ≠ predictive (DLA) and steering

In [ ]:
d = load('eval/results/dla/hellaswag.json')
dla = pd.DataFrame([{ 'feature': r['feature'], 'predictive_AUROC': round(r['predictive_AUROC'],3),
  'causal_Δ(nats)': r['causal_delta_error'], 'DLA_mag': round(r['dla_magnitude'],3) } for r in d['features']]) if d else pd.DataFrame()
print('mean predictive AUROC of top causal features:', round(d['mean_predictive_AUROC'],3)) if d else None
st = load('eval/results/steering/squad.json')
if st: print('steering max |output shift| =', round(st['max_abs_shift_nats'],4), 'nats over', st['n_eval'], 'examples')
dla

## §5.11 What carries P1 (coefficient / SHAP recipe)

In [ ]:
d = load('eval/results/baseline_shap/squad_l12.json')
print('P1 full AUROC =', round(d['P1_full_AUROC'],3), ' recipe (top 3):', d['recipe']) if d else None
pd.DataFrame(d['ranked_features']) if d else 'missing'